In [ ]:
# =============================================================================
# NOTEBOOK: catalog_ddl
# PURPOSE:  Provision the vinoworld catalog, schemas, volumes, and all
#           managed tables across the audit, bronze, and silver layers.
#           Safe to re-run — all statements use CREATE IF NOT EXISTS.
# DEPENDS ON: /Workspace/Shared/notebook_init, /Workspace/Shared/catalog_setup
# RUN ONCE:   Re-run individual cells to re-provision a specific layer.
# =============================================================================
%run "/Workspace/Shared/notebook_init"

In [ ]:
# Cell 2 — Imports
# notebook_init already appended /Workspace/Shared to sys.path
from catalog_setup import (
    create_catalog,
    create_schemas,
    create_volume_schema,
    create_volumes,
    create_audit_tables,
    create_bronze_tables,
    create_silver_tables,
)

In [ ]:
# Cell 3 — Catalog
result = create_catalog(spark, CATALOG)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 4 — Medallion schemas and volume schema
result = create_schemas(spark, [BRONZE, SILVER, GOLD, AUDIT])
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

result = create_volume_schema(spark, CATALOG)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 5 — Volumes
# Add a dict here for each new source system volume.
VOLUME_DEFINITIONS = [
    {"name": "arancione",   "needs_archive": True},
    {"name": "celeste",     "needs_archive": True},
    {"name": "verde",       "needs_archive": True},
    {"name": "productdata", "needs_archive": True},
    {"name": "masterdata",  "needs_archive": False},
]

result = create_volumes(spark, dbutils, CATALOG, VOLUME_DEFINITIONS)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 6 — Audit tables
result = create_audit_tables(spark, AUDIT)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 7 — Bronze tables
result = create_bronze_tables(spark, BRONZE)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 8 — Silver tables
result = create_silver_tables(spark, SILVER)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])